# d52: One Substrate + One Catalyst, Step-By-Step TS Assembly

This notebook is the short, direct version of the screen/TS guess workflow. It
uses exactly one substrate SMILES and one catalyst SMILES, then walks through
how FRUST turns them into a transition-state guess.

Think of it like an IKEA instruction sheet:

```text
part A: substrate SMILES
part B: catalyst SMILES
part C: chosen reactive C-H position
part D: TS template roles
part E: optional fragment such as HBpin

assemble -> place on template -> inspect distances/angles -> hand to Stepper
```

The detailed build below focuses on `TS3` because it includes the most useful
pieces to understand: catalyst, substrate, catalyst B-H, HBpin, template role
coordinates, hard anchors, soft frame anchors, and row-level constraints. At the
end, the same one-substrate/one-catalyst system is used to generate `TS1`-`TS4`.


## How To Inspect The 3D Panels

The 3D panels use FRUST's `MolTo3DGrid` viewer.

- Click an atom to toggle its atom index label.
- Ctrl-click two atoms to measure a distance.
- Shift-click three atoms to measure an angle.

Highlighted atoms are the atoms being discussed in that step. Labels such as
`cat_B:16` mean: this atom is playing the chemical role `cat_B`, and its atom
index in the current molecule is `16`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import SVG, display
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Geometry import Point3D

import frust as ft
from frust.constraints import render_xtb_constraints
from frust.tsguess.assembly import (
    _add_final_ts_bonds,
    _allowed_contact_pairs,
    _assemble_system_mol,
    _connectivity_bonds,
    _hard_placement_atom_indices,
    _placement_coord_map,
)
from frust.tsguess.embedding import embed_with_coord_map
from frust.tsguess.matching import (
    hydrogens_on_atom,
    match_catalyst_roles,
    mol_from_smiles,
    substrate_hydrogen_for_rpos,
)
from frust.tsguess.specs import BUILTIN_TS_SPECS
from frust.vis.molecules import _row_to_mol

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

ROOT = Path.cwd()
if ROOT.name == "dev":
    ROOT = ROOT.parent

print(f"Working from: {ROOT}")


In [ ]:
ROLE_COLORS = {
    "cat_B": (0.1, 0.75, 0.15),
    "cat_N": (0.35, 0.35, 0.95),
    "substrate_C": (0.95, 0.55, 0.05),
    "cat_H": (0.95, 0.95, 0.95),
    "transfer_H": (1.0, 0.15, 0.15),
    "n_transfer_H": (0.7, 0.25, 0.95),
    "pin_B": (0.0, 0.75, 0.75),
}
HARD_COLOR = (1.0, 0.55, 0.05)
SOFT_COLOR = (0.1, 0.75, 0.95)


def mol_copy(mol):
    return Chem.Mol(mol)


def embed_copy(mol, seed=0xF00D):
    out = Chem.Mol(mol)
    if out.GetNumConformers() == 0:
        AllChem.EmbedMolecule(out, randomSeed=seed, useRandomCoords=False)
    return out


def label_atoms(mol, labels):
    out = Chem.Mol(mol)
    for atom_idx, label in labels.items():
        out.GetAtomWithIdx(int(atom_idx)).SetProp("atomNote", str(label))
    return out


def label_roles(mol, roles):
    return label_atoms(
        mol,
        {int(atom_idx): f"{role}:{int(atom_idx)}" for role, atom_idx in roles.items()},
    )


def draw_2d(mol, title="", highlights=None, labels=None, width=650, height=430):
    draw_mol = Chem.Mol(mol)
    if labels:
        for atom_idx, label in labels.items():
            draw_mol.GetAtomWithIdx(int(atom_idx)).SetProp("atomNote", str(label))
    rdDepictor.SetPreferCoordGen(True)
    rdDepictor.Compute2DCoords(draw_mol)

    atom_colors = {}
    if highlights:
        for atom_idx, color in highlights.items():
            atom_colors[int(atom_idx)] = tuple(color[:3])
    highlight_atoms = list(atom_colors)

    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.addAtomIndices = True
    opts.annotationFontScale = 0.75
    opts.bondLineWidth = 2
    drawer.DrawMolecule(
        draw_mol,
        legend=title,
        highlightAtoms=highlight_atoms,
        highlightAtomColors=atom_colors,
    )
    drawer.FinishDrawing()
    display(SVG(drawer.GetDrawingText()))


def show_3d(mols, legends, highlights=None, columns=2, linked=True, size=(520, 430)):
    if not isinstance(mols, list):
        mols = [mols]
    if highlights is None:
        highlights = [[] for _ in mols]
    ft.MolTo3DGrid(
        [embed_copy(mol) for mol in mols],
        legends=legends,
        highlightAtoms=highlights,
        show_labels=True,
        show_confs=False,
        columns=columns,
        linked=linked,
        cell_size=size,
        kekulize=True,
        show_charges=True,
    )


def row_to_role_mol(row, coord_col="coords_embedded"):
    mol = _row_to_mol(row, row["atoms"], row[coord_col])
    return label_roles(mol, row["constraint_roles"])


def distance(coords, a, b):
    return float(np.linalg.norm(coords[int(a)] - coords[int(b)]))


def angle(coords, a, b, c):
    ba = coords[int(a)] - coords[int(b)]
    bc = coords[int(c)] - coords[int(b)]
    cosine = float(np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc)))
    cosine = max(-1.0, min(1.0, cosine))
    return float(np.degrees(np.arccos(cosine)))


def constraint_table(row, coord_col="coords_embedded"):
    roles = row["constraint_roles"]
    coords = np.asarray(row[coord_col], dtype=float)
    records = []
    for item in row["constraint_spec"]:
        role_names = list(item["roles"])
        atom_ids = [int(roles[role]) for role in role_names]
        if item["kind"] == "distance":
            measured = distance(coords, atom_ids[0], atom_ids[1])
        else:
            measured = angle(coords, atom_ids[0], atom_ids[1], atom_ids[2])
        records.append(
            {
                "kind": item["kind"],
                "roles": "-".join(role_names),
                "atoms": "-".join(str(i) for i in atom_ids),
                "target": round(float(item["value"]), 3),
                "measured": round(measured, 3),
                "delta": round(measured - float(item["value"]), 3),
            }
        )
    return pd.DataFrame(records)


def role_table(roles, mol):
    return pd.DataFrame(
        [
            {
                "role": role,
                "atom_index": int(atom_idx),
                "element": mol.GetAtomWithIdx(int(atom_idx)).GetSymbol(),
            }
            for role, atom_idx in sorted(roles.items())
        ]
    )


def highlight_from_roles(roles):
    return {
        int(atom_idx): ROLE_COLORS.get(role, (0.9, 0.6, 0.0))
        for role, atom_idx in roles.items()
    }


## Step 1. Put The Two Parts On The Table

Use one substrate and one catalyst. Nothing chemical has happened yet; this is
just the input.


In [ ]:
substrate_smiles = "CN1C=CC=C1"
catalyst_smiles = "CC1(C)CCCC(C)(C)N1C2=CC=CC=C2B"
substrate_name = "n_methyl_pyrrole"
catalyst_name = "tmp_bcat"
chosen_rpos = 2

pd.DataFrame(
    [
        {"part": "substrate", "name": substrate_name, "smiles": substrate_smiles},
        {"part": "catalyst", "name": catalyst_name, "smiles": catalyst_smiles},
    ]
)


2D view: atom indices are shown. The chosen substrate `rpos` will be atom `2`.


In [ ]:
substrate_mol = mol_from_smiles(substrate_smiles, label=substrate_name)
catalyst_mol = mol_from_smiles(catalyst_smiles, label=catalyst_name)

draw_2d(substrate_mol, title="substrate: atom 2 will be the reactive C-H")
draw_2d(catalyst_mol, title="catalyst: FRUST will search for B-aryl-N")


3D view: these are just ordinary RDKit embeddings of the two starting parts.
They are not TS structures yet.


In [ ]:
show_3d(
    [substrate_mol, catalyst_mol],
    legends=["substrate input", "catalyst input"],
    columns=2,
)


## Step 2. Mark The Substrate C-H To Remove

The substrate `rpos` points to the aromatic carbon that will become
`substrate_C` in the TS guess. FRUST adds explicit hydrogens so it can identify
and remove the attached H.


In [ ]:
substrate_h = Chem.AddHs(substrate_mol)
leaving_h = substrate_hydrogen_for_rpos(
    substrate_h,
    chosen_rpos,
    substrate_name=substrate_name,
)
substrate_labels = {
    chosen_rpos: f"rpos/substrate_C:{chosen_rpos}",
    leaving_h: f"H removed:{leaving_h}",
}
substrate_highlights = {
    chosen_rpos: ROLE_COLORS["substrate_C"],
    leaving_h: ROLE_COLORS["transfer_H"],
}

pd.DataFrame(
    [
        {"item": "chosen rpos", "atom_index": chosen_rpos},
        {"item": "attached H to remove", "atom_index": leaving_h},
    ]
)


In [ ]:
draw_2d(
    substrate_h,
    title="substrate with explicit H: atom 2 keeps the carbon, its H is removed",
    highlights=substrate_highlights,
    labels=substrate_labels,
)
show_3d(
    label_atoms(substrate_h, substrate_labels),
    legends=["substrate before H removal"],
    highlights=[[chosen_rpos, leaving_h]],
    columns=1,
)


Now remove that hydrogen. This is how the substrate is prepared for assembly:
the selected carbon becomes the open `substrate_C` role.


In [ ]:
editable = Chem.RWMol(substrate_h)
editable.RemoveAtom(int(leaving_h))
substrate_open = editable.GetMol()
substrate_open.UpdatePropertyCache(strict=False)
adjusted_substrate_c = chosen_rpos - int(leaving_h < chosen_rpos)

open_labels = {adjusted_substrate_c: f"substrate_C:{adjusted_substrate_c}"}
open_highlights = {adjusted_substrate_c: ROLE_COLORS["substrate_C"]}

pd.DataFrame(
    [
        {"before H removal rpos": chosen_rpos, "after H removal substrate_C": adjusted_substrate_c}
    ]
)


In [ ]:
draw_2d(
    substrate_open,
    title="prepared substrate: selected H removed",
    highlights=open_highlights,
    labels=open_labels,
)
show_3d(
    label_atoms(substrate_open, open_labels),
    legends=["prepared substrate fragment"],
    highlights=[[adjusted_substrate_c]],
    columns=1,
)


## Step 3. Find The Catalyst B-Aryl-N Scaffold

This is the central new trick. The catalyst is not assumed to have fixed atom
indices. FRUST finds the B-aryl-N scaffold and assigns `cat_B` and `cat_N`.


In [ ]:
catalyst_h = Chem.AddHs(catalyst_mol)
cat_roles = match_catalyst_roles(catalyst_h, catalyst_name=catalyst_name)
cat_b_hydrogens = hydrogens_on_atom(
    catalyst_h,
    cat_roles["cat_B"],
    role="cat_B",
    minimum=2,
)
cat_labels = {
    cat_roles["cat_B"]: f"cat_B:{cat_roles['cat_B']}",
    cat_roles["cat_N"]: f"cat_N:{cat_roles['cat_N']}",
}
cat_labels.update({h: f"B-H:{h}" for h in cat_b_hydrogens})
cat_highlights = {
    cat_roles["cat_B"]: ROLE_COLORS["cat_B"],
    cat_roles["cat_N"]: ROLE_COLORS["cat_N"],
}
cat_highlights.update({h: ROLE_COLORS["cat_H"] for h in cat_b_hydrogens})

role_table(cat_roles, catalyst_h)


In [ ]:
draw_2d(
    catalyst_h,
    title="catalyst with dynamic cat_B/cat_N roles and B-H atoms",
    highlights=cat_highlights,
    labels=cat_labels,
)
show_3d(
    label_atoms(catalyst_h, cat_labels),
    legends=["catalyst roles from B-aryl-N match"],
    highlights=[[cat_roles["cat_B"], cat_roles["cat_N"], *cat_b_hydrogens]],
    columns=1,
)


This is how the new code generalizes beyond one fixed catalyst XYZ template:
it asks which atoms play `cat_B` and `cat_N` in this catalyst SMILES.


## Step 4. Make A One-Row Screen

Now put the two parts into the public screen workflow. `screen.read` normalizes
the component table. `screen.expand` creates the one substrate-catalyst system.


In [ ]:
components = ft.screen.read(
    pd.DataFrame(
        {
            "role": ["substrate", "catalyst"],
            "smiles": [substrate_smiles, catalyst_smiles],
            "compound_name": [substrate_name, catalyst_name],
            "rpos": [str(chosen_rpos), ""],
        }
    )
)
systems = ft.screen.expand(components)

components


In [ ]:
systems


## Step 5. Choose The Instruction Sheet: TS3

For the furniture-style build, use `TS3`. This spec says which chemical roles
are needed, where their template coordinates are, and which role-based distances
and angles should be constrained.


In [ ]:
ts_type = "TS3"
spec = BUILTIN_TS_SPECS[ts_type]

pd.DataFrame(
    [
        {
            "ts_type": spec.name,
            "spec_id": spec.spec_id,
            "extra_fragment": spec.extra_fragment,
            "role_coordinates": tuple(spec.role_coordinates),
            "constraint_order": spec.constraint_order,
            "n_constraints": len(spec.constraints),
        }
    ]
)


In [ ]:
pd.DataFrame([entry.as_dict() for entry in spec.constraints])


2D instruction graphic: this is not a molecule. It is a simple RDKit drawing of
the TS3 role/constraint graph. Each node is a role, and each drawn edge is a
role-based distance constraint.


In [ ]:
def role_constraint_graph(spec):
    editable = Chem.RWMol()
    role_to_idx = {}
    for role in spec.role_coordinates:
        atom = Chem.Atom(0)
        atom.SetProp("atomNote", role)
        role_to_idx[role] = editable.AddAtom(atom)
    for entry in spec.constraints:
        if entry.kind != "distance":
            continue
        left, right = entry.roles[:2]
        i, j = role_to_idx[left], role_to_idx[right]
        if editable.GetBondBetweenAtoms(i, j) is None:
            editable.AddBond(i, j, Chem.BondType.SINGLE)
    graph = editable.GetMol()
    graph.UpdatePropertyCache(strict=False)
    return graph, role_to_idx

constraint_graph, graph_roles = role_constraint_graph(spec)
draw_2d(
    constraint_graph,
    title="TS3 role constraint graph: edges are constrained distances",
    highlights={idx: ROLE_COLORS.get(role, (0.8, 0.8, 0.8)) for role, idx in graph_roles.items()},
    labels={idx: role for role, idx in graph_roles.items()},
)


3D instruction graphic: these are the TS3 role coordinates from the built-in
spec. This is the target core that the real atoms will be placed onto.


In [ ]:
def role_template_mol(spec):
    symbol_by_role = {
        "cat_B": "B",
        "cat_N": "N",
        "substrate_C": "C",
        "pin_B": "B",
        "cat_H": "H",
        "transfer_H": "H",
        "n_transfer_H": "H",
    }
    editable = Chem.RWMol()
    role_to_idx = {}
    for role in spec.role_coordinates:
        atom = Chem.Atom(symbol_by_role.get(role, "C"))
        atom.SetProp("atomNote", role)
        role_to_idx[role] = editable.AddAtom(atom)
    mol = editable.GetMol()
    mol.UpdatePropertyCache(strict=False)
    conf = Chem.Conformer(mol.GetNumAtoms())
    for role, idx in role_to_idx.items():
        x, y, z = spec.role_coordinates[role]
        conf.SetAtomPosition(int(idx), Point3D(float(x), float(y), float(z)))
    mol.AddConformer(conf, assignId=True)
    return mol, role_to_idx

template_mol, template_roles = role_template_mol(spec)
show_3d(
    template_mol,
    legends=["TS3 role-coordinate template"],
    highlights=[list(template_roles.values())],
    columns=1,
)


## Step 6. Assemble The Parts Without Placing Them Yet

Now use the same assembly helper that `ft.screen.create_ts_guesses(...)` uses
under the hood. At this point, the molecule is a collection of disconnected
fragments: trimmed catalyst, dehydrogenated substrate, and HBpin.

No TS geometry has been imposed yet.


In [ ]:
system = systems.iloc[0]
assembled, roles = _assemble_system_mol(system, spec, chosen_rpos)
assembled_labeled = label_roles(assembled, roles)

role_table(roles, assembled)


In [ ]:
fragment_summary = []
for frag_id, atom_indices in enumerate(Chem.GetMolFrags(assembled)):
    elements = [assembled.GetAtomWithIdx(i).GetSymbol() for i in atom_indices]
    fragment_summary.append(
        {
            "fragment": frag_id,
            "n_atoms": len(atom_indices),
            "elements_preview": "".join(elements[:12]),
        }
    )

pd.DataFrame(fragment_summary)


In [ ]:
draw_2d(
    assembled_labeled,
    title="assembled TS3 graph before template placement",
    highlights=highlight_from_roles(roles),
    labels={idx: f"{role}:{idx}" for role, idx in roles.items()},
)
show_3d(
    assembled_labeled,
    legends=["assembled fragments before template placement"],
    highlights=[list(roles.values())],
    columns=1,
)


The 3D view above is just a generic embedding of disconnected fragments. The
next step is where the TS template geometry is applied.


## Step 7. Build The Placement Map

The placement map tells RDKit where selected atoms should go.

- Hard anchors are role atoms. They are snapped to the template exactly.
- Soft frame anchors are nearby substrate/HBpin atoms. They orient larger
  fragments without pretending every atom is part of the TS core.


In [ ]:
coord_map = _placement_coord_map(assembled, spec, roles)
hard_anchor_indices = _hard_placement_atom_indices(spec, roles)

placement_records = []
role_by_atom = {int(v): k for k, v in roles.items()}
for atom_idx, xyz in coord_map.items():
    atom_idx = int(atom_idx)
    placement_records.append(
        {
            "atom_index": atom_idx,
            "element": assembled.GetAtomWithIdx(atom_idx).GetSymbol(),
            "anchor_type": "hard role" if atom_idx in hard_anchor_indices else "soft frame",
            "role_if_any": role_by_atom.get(atom_idx, ""),
            "target_xyz": tuple(round(float(v), 3) for v in xyz),
        }
    )

pd.DataFrame(placement_records).sort_values(["anchor_type", "atom_index"])


In [ ]:
anchor_highlights = {
    int(atom_idx): (HARD_COLOR if int(atom_idx) in hard_anchor_indices else SOFT_COLOR)
    for atom_idx in coord_map
}
anchor_labels = {
    int(atom_idx): role_by_atom.get(int(atom_idx), f"soft:{int(atom_idx)}")
    for atom_idx in coord_map
}

draw_2d(
    assembled,
    title="orange = hard role anchors; cyan = soft frame anchors",
    highlights=anchor_highlights,
    labels=anchor_labels,
)
show_3d(
    label_atoms(assembled, anchor_labels),
    legends=["anchor atoms before placement"],
    highlights=[list(coord_map)],
    columns=1,
)


## Step 8. Embed And Place The Fragments

This is the RDKit constrained-embedding part. Because the assembled molecule has
disconnected fragments, FRUST embeds fragments and then rigidly places each
fragment from its anchors. After placement, it rotates under-anchored fragments
to reduce clashes.


In [ ]:
embedded, cids = embed_with_coord_map(
    assembled,
    coord_map,
    n_confs=1,
    n_cores=1,
    allowed_contact_pairs=_allowed_contact_pairs(spec, roles),
    snap_atom_indices=hard_anchor_indices,
)
embedded = _add_final_ts_bonds(embedded, spec, roles)
embedded_labeled = label_roles(embedded, roles)

print(f"Generated conformer IDs: {cids}")
print(f"Stored connectivity bonds after final TS graph step: {len(_connectivity_bonds(embedded))}")


In [ ]:
draw_2d(
    embedded_labeled,
    title="final TS3 stored graph after template placement",
    highlights=highlight_from_roles(roles),
    labels={idx: f"{role}:{idx}" for role, idx in roles.items()},
)
show_3d(
    embedded_labeled,
    legends=["final generated TS3 guess"],
    highlights=[list(roles.values())],
    columns=1,
)


Now the geometry is meaningful: the role atoms have been placed on the TS3 core,
and the rest of the catalyst/substrate/HBpin fragments follow those anchors.


## Step 9. Compare With The Public Row

The public API wraps the same process and returns a dataframe row. This row is
what later goes into `Stepper`.


In [ ]:
ts_guesses = ft.screen.create_ts_guesses(
    systems,
    ts_types=["TS3"],
    n_confs=1,
)
ts3_row = ts_guesses["TS3"].iloc[0]

pd.DataFrame(
    [
        {
            "column": col,
            "value": ts3_row[col],
        }
        for col in [
            "custom_name",
            "structure_type",
            "system_name",
            "substrate_name",
            "catalyst_name",
            "rpos",
            "cid",
            "ts_spec_id",
        ]
    ]
)


In [ ]:
role_table(ts3_row["constraint_roles"], _row_to_mol(ts3_row, ts3_row["atoms"], ts3_row["coords_embedded"]))


## Step 10. Measure The Core Distances And Angles

This is the quantitative version of Ctrl-click and Shift-click in the 3D viewer.
The same atom indices are shown in the `atoms` column.


In [ ]:
constraint_table(ts3_row)


The `target` values are the template constraints. The `measured` values are from
the generated 3D coordinates. Small deltas mean the generated guess preserved
the intended core.


In [ ]:
print(render_xtb_constraints(ts3_row))


That printed block is what `Stepper(..., constraint=True)` passes to xTB for
this row. The old workflow needed `step_type` plus `constraint_atoms`; the new
row carries the chemical role map and the constraint spec directly.


## Step 11. Show The Same One-System Build For TS1-TS4

The same substrate/catalyst/rpos can be passed through all built-in TS specs.
Each row has different role atoms and constraints, but the assembly logic is the
same.


In [ ]:
all_ts = ft.screen.create_ts_guesses(
    systems,
    ts_types=["TS1", "TS2", "TS3", "TS4"],
    n_confs=1,
)
rows = [all_ts[ts].iloc[0] for ts in ["TS1", "TS2", "TS3", "TS4"]]

pd.DataFrame(
    [
        {
            "ts_type": row["structure_type"],
            "roles": tuple(row["constraint_roles"]),
            "n_constraints": len(row["constraint_spec"]),
            "n_atoms": len(row["atoms"]),
        }
        for row in rows
    ]
)


In [ ]:
final_mols = [row_to_role_mol(row) for row in rows]
final_highlights = [list(row["constraint_roles"].values()) for row in rows]
final_legends = [
    "TS1: C-H activation",
    "TS2: H-H motif",
    "TS3: HBpin B-H motif",
    "TS4: catalyst B-H motif",
]

ft.MolTo3DGrid(
    final_mols,
    legends=final_legends,
    highlightAtoms=final_highlights,
    show_labels=True,
    show_confs=False,
    columns=2,
    linked=True,
    cell_size=(540, 460),
    kekulize=True,
    show_charges=True,
)


## Step 12. The Whole Assembly In One Sentence

FRUST starts with two SMILES, finds chemically meaningful atom roles, prepares
the necessary fragments, places those role atoms on a built-in TS template,
stores the resulting coordinates and graph, and writes role-based constraints
into the row so `Stepper` can optimize the guess without knowing any fixed atom
indices.

That is why this now behaves like the old transformer in spirit, but can expand
over related catalysts: the template is no longer tied to one fixed XYZ atom
order.
